In [35]:
import gdown

file_id = "1Uc-ZvLWaWY9Ua5Y5lEsP8FTOz9UarTcU"
url = f"https://drive.google.com/uc?id={file_id}"

gdown.download(url, "employees2.csv", quiet=False)


Downloading...
From: https://drive.google.com/uc?id=1Uc-ZvLWaWY9Ua5Y5lEsP8FTOz9UarTcU
To: C:\WINDOWS\system32\AI\employees2.csv
100%|██████████| 3.34k/3.34k [00:00<00:00, 3.62MB/s]


'employees2.csv'

In [36]:
import pandas as pd
import numpy as np

In [37]:
# 1. Charger le fichier
df = pd.read_csv("employees2.csv")
print(df.head())

   Unnamed: 0  ID     Name   Age   Salary Department  Years_Experience Remote
0           0   1  Othmane   NaN      NaN         IT                 0    Yes
1           1   2    Ikram  22.0      NaN         HR                23    Yes
2           2   3      Bob   NaN      NaN         HR                26    Yes
3           3   4    Zineb  59.0  73665.0         HR                 3     No
4           4   5    David  57.0  59325.0    Finance                33     No


In [38]:
# 3. Vérifier les types de données
print(df.dtypes)

Unnamed: 0            int64
ID                    int64
Name                    str
Age                 float64
Salary              float64
Department              str
Years_Experience      int64
Remote                  str
dtype: object


In [39]:
# 4. Valeurs manquantes
print(df.isnull().sum())

Unnamed: 0           0
ID                   0
Name                 0
Age                 49
Salary              44
Department           0
Years_Experience     0
Remote               0
dtype: int64


In [40]:
# Nettoyage des données
# Remplacer les valeurs manquantes dans Age par la médiane
df["Age"] = df["Age"].fillna(df["Age"].median())

In [41]:
print(df.columns)

Index(['Unnamed: 0', 'ID', 'Name', 'Age', 'Salary', 'Department',
       'Years_Experience', 'Remote'],
      dtype='str')


In [42]:
# Remplir Salaire par la moyenne du département
df["Salary"] = df.groupby("Department")["Salary"].transform(
    lambda x: x.fillna(x.mean())
)

In [43]:
# Convertir les colonnes numériques
df["Age"] = df["Age"].astype(int)
df["Salary"] = df["Salary"].astype(float)
df["Years_Experience"] = df["Years_Experience"].astype(int)

In [44]:
df["Remote"] = df["Remote"].replace({"Yes": "Oui", "No": "Non"})

In [45]:
def categoriser(exp):
    if exp < 3:
        return "Junior"
    elif exp <= 7:
        return "Intermédiaire"
    elif exp <= 15:
        return "Senior"
    else:
        return "Expert"

df["Anciennete_Categorie"] = df["Years_Experience"].apply(categoriser)

In [46]:
print(df["Salary"].mean())

64504.14296153846


In [47]:
print(df.loc[df["Salary"].idxmax()])

Unnamed: 0                     27
ID                             28
Name                        Jalil
Age                            45
Salary                    84685.0
Department              Marketing
Years_Experience               28
Remote                        Oui
Anciennete_Categorie       Expert
Name: 27, dtype: object


In [48]:
print(df.groupby("Department")["Salary"].mean())

Department
Finance      57836.583333
HR           61659.733333
IT           69930.000000
Logistics    64544.750000
Marketing    70395.153846
Name: Salary, dtype: float64


In [49]:
print(df.groupby(["Department", "Remote"]).size())

Department  Remote
Finance     Non       13
            Oui        7
HR          Non       15
            Oui       13
IT          Non        3
            Oui        9
Logistics   Non        7
            Oui        8
Marketing   Non        7
            Oui       18
dtype: int64


In [50]:
pivot1 = pd.pivot_table(
    df,
    values="Salary",
    index="Department",
    columns="Remote",
    aggfunc="mean"
)
print(pivot1)

Remote               Non           Oui
Department                            
Finance     60964.762821  52027.107143
HR          62817.608889  60323.723077
IT          65695.000000  71341.666667
Logistics   65080.750000  64075.750000
Marketing   69812.538462  70621.726496


In [51]:
df["Groupe_Age"] = pd.cut(
    df["Age"],
    bins=[0, 30, 45, 60, 100],
    labels=["Jeune", "Adulte", "Senior", "Très Senior"]
)

pivot2 = pd.pivot_table(
    df,
    values="Years_Experience",
    index="Groupe_Age",
    columns="Department",
    aggfunc="mean"
)
print(pivot2)

Department    Finance       HR         IT  Logistics  Marketing
Groupe_Age                                                     
Jeune             NaN  22.2500  13.000000  24.666667  18.000000
Adulte      19.588235  15.0625  17.285714  13.666667  20.625000
Senior      16.666667  20.7500  20.000000  20.000000  19.857143


In [52]:
df["Performance"] = np.where(
    df["Salary"] < 60000, "Bon",
    np.where(df["Salary"] < 80000, "Moyen", "Haut")
)

In [53]:
conditions = [
    (df["Age"] < 40) & (df["Years_Experience"] < 5),
    (df["Age"] < 40) & (df["Years_Experience"] >= 5),
    (df["Age"] >= 40) & (df["Years_Experience"] < 5),
    (df["Age"] >= 40) & (df["Years_Experience"] >= 5)
]

choix = [
    "Jeune & Nouveau",
    "Jeune & Expérimenté",
    "Senior & Nouveau",
    "Senior & Expérimenté"
]

df["Profil"] = np.select(conditions, choix, default="Inconnu")

In [54]:
moy_dep = df.groupby("Department")["Salary"].transform("mean")
df["Diff_Salary_Department"] = df["Salary"] - moy_dep